In [4]:
import os
from dotenv import load_dotenv
from openai import OpenAI

import gradio as gr
import warnings
warnings.filterwarnings("ignore")

In [5]:
load_dotenv(override=True)
openai_api_key = os.getenv("OPENAI_API_KEY")

In [6]:
openai = OpenAI()

In [8]:
system_prompt = "You are a helpful assistant."

def message_gpt(prompt):
    messages = [
        {"role": "system", "content": system_prompt},
        {"role":"user", "content": prompt}
    ]
    response = openai.chat.completions.create(model='gpt-4.1-mini', messages=messages)
    return response.choices[0].message.content

## Basics of Gradio

In [6]:
# define the callback
def shout(text):
    print(f"Shout has been called with the input: {text}")
    return text.upper()

In [ ]:
# Keep name aligned with behavior (this enforces LIGHT theme)
force_light_mode = """
function refresh() {
    const url = new URL(window.location);
    if (url.searchParams.get('__theme') !== 'dark') {
        url.searchParams.set('__theme', 'dark');
        window.location.href = url.href;
    }
}
refresh();
"""

message_input = gr.Textbox(label="Your message:", info="Enter a message to be shouted", lines=7)
message_output = gr.Textbox(label="Response:", lines=8)

view = gr.Interface(
    fn=shout,
    title="Shout",
    inputs=[message_input],
    outputs=[message_output],
    examples=["hello", "howdy"],
    flagging_mode="never",
    js=force_light_mode
)

view.launch()

* Running on local URL:  http://127.0.0.1:7869
* To create a public link, set `share=True` in `launch()`.


## Talk to GPT from within Gradio

In [19]:
# Keep name aligned with behavior (this enforces LIGHT theme)
force_light_mode = """
function refresh() {
    const url = new URL(window.location);
    if (url.searchParams.get('__theme') !== 'dark') {
        url.searchParams.set('__theme', 'dark');
        window.location.href = url.href;
    }
}
refresh();
"""

message_input = gr.Textbox(label="Your message:", info="Enter a message for GPT 4.1 mini", lines=7)
message_output = gr.Textbox(label="Response:", lines=8)

view = gr.Interface(
    fn=message_gpt,
    title="GPT",
    inputs=[message_input],
    outputs=[message_output],
    examples=["hello", "howdy"],
    flagging_mode="never",
    js=force_light_mode
)

view.launch()

* Running on local URL:  http://127.0.0.1:7870
* To create a public link, set `share=True` in `launch()`.


Exception in callback _ProactorBasePipeTransport._call_connection_lost()
handle: <Handle _ProactorBasePipeTransport._call_connection_lost()>
Traceback (most recent call last):
  File "C:\Users\abhas\AppData\Local\Programs\Python\Python314\Lib\asyncio\events.py", line 94, in _run
    self._context.run(self._callback, *self._args)
    ~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\abhas\AppData\Local\Programs\Python\Python314\Lib\asyncio\proactor_events.py", line 165, in _call_connection_lost
    self._sock.shutdown(socket.SHUT_RDWR)
    ~~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^
ConnectionResetError: [WinError 10054] An existing connection was forcibly closed by the remote host


## Add Markdown to output

In [10]:
# Keep name aligned with behavior (this enforces LIGHT theme)
force_light_mode = """
function refresh() {
    const url = new URL(window.location);
    if (url.searchParams.get('__theme') !== 'dark') {
        url.searchParams.set('__theme', 'dark');
        window.location.href = url.href;
    }
}
refresh();
"""
system_message = "You are a helpful assistant that responds in markdown without code blocks"
message_input = gr.Textbox(label="Your message:", info="Enter a message for GPT 4.1 mini", lines=7)
message_output = gr.Markdown(label="Response:")

view = gr.Interface(
    fn=message_gpt,
    title="GPT",
    inputs=[message_input],
    outputs=[message_output],
    examples=["Explain the transformer architecture to a layperson in less than 100 words",
              "Explain the transformer architecture to an aspiring AI enginerr in less than 100 words"],
    flagging_mode="never",
    js=force_light_mode
)

view.launch()

* Running on local URL:  http://127.0.0.1:7861
* To create a public link, set `share=True` in `launch()`.


## Add Streaming

In [11]:
def stream_gpt(prompt):
    """replace message_gpt with a generator function"""
    messages = [
        {"role":"system", "content":system_message},
        {"role":"user","content":prompt}
    ]

    stream = openai.chat.completions.create(
        model="gpt-4.1-mini",
        messages=messages,
        stream=True
    )

    result = ""
    for chunk in stream:
        result += chunk.choices[0].delta.content or ""
        yield result

In [ ]:
# Keep name aligned with behavior (this enforces LIGHT theme)
force_light_mode = """
function refresh() {
    const url = new URL(window.location);
    if (url.searchParams.get('__theme') !== 'dark') {
        url.searchParams.set('__theme', 'dark');
        window.location.href = url.href;
    }
}
refresh();
"""
system_message = "You are a helpful assistant that responds in markdown without code blocks"
message_input = gr.Textbox(label="Your message:", info="Enter a message for GPT 4.1 mini", lines=7)
message_output = gr.Markdown(label="Response:")

view = gr.Interface(
    fn=stream_gpt,
    title="GPT",
    inputs=[message_input],
    outputs=[message_output],
    examples=["Explain the transformer architecture to a layperson in less than 100 words",
              "Explain the transformer architecture to an aspiring AI enginerr in less than 100 words"],
    flagging_mode="never",
    js=force_light_mode
)

view.launch()

* Running on local URL:  http://127.0.0.1:7873
* To create a public link, set `share=True` in `launch()`.


Exception in callback _ProactorBasePipeTransport._call_connection_lost()
handle: <Handle _ProactorBasePipeTransport._call_connection_lost()>
Traceback (most recent call last):
  File "C:\Users\abhas\AppData\Local\Programs\Python\Python314\Lib\asyncio\events.py", line 94, in _run
    self._context.run(self._callback, *self._args)
    ~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\abhas\AppData\Local\Programs\Python\Python314\Lib\asyncio\proactor_events.py", line 165, in _call_connection_lost
    self._sock.shutdown(socket.SHUT_RDWR)
    ~~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^
ConnectionResetError: [WinError 10054] An existing connection was forcibly closed by the remote host


: 

## Adding Dropdowns for selection

In [13]:
def stream_model(prompt, model):
    if model=='GPT':
        result = stream_gpt(prompt=prompt)
    else:
        raise ValueError("Invalid Model")
    yield from result

In [21]:
message_input = gr.Textbox(label="Your message", info="Enter a message for the LLM", lines=7)
model_selector = gr.Dropdown(["GPT", "Claude"], label="Select model", value="GPT")
message_output = gr.Markdown(label="Response:")


view = gr.Interface(
    fn=stream_model,
    title="LLMs",
    inputs=[message_input, model_selector],
    outputs=[message_output],
    examples=[
        ["Howdy", "GPT"],
        ["Hello", "Claude"]
    ],
    flagging_mode="never"
)

view.launch()

* Running on local URL:  http://127.0.0.1:7865
* To create a public link, set `share=True` in `launch()`.


Traceback (most recent call last):
  File "c:\Users\abhas\Desktop\Dev\CodeBase\01. LLM Engineering Basics\.venv\Lib\site-packages\gradio\queueing.py", line 856, in process_events
    response = await route_utils.call_process_api(
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
    ...<5 lines>...
    )
    ^
  File "c:\Users\abhas\Desktop\Dev\CodeBase\01. LLM Engineering Basics\.venv\Lib\site-packages\gradio\route_utils.py", line 358, in call_process_api
    output = await app.get_blocks().process_api(
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
    ...<11 lines>...
    )
    ^
  File "c:\Users\abhas\Desktop\Dev\CodeBase\01. LLM Engineering Basics\.venv\Lib\site-packages\gradio\blocks.py", line 2179, in process_api
    result = await self.call_function(
             ^^^^^^^^^^^^^^^^^^^^^^^^^
    ...<8 lines>...
    )
    ^
  File "c:\Users\abhas\Desktop\Dev\CodeBase\01. LLM Engineering Basics\.venv\Lib\site-packages\gradio\blocks.py", line 1648, in call_function
    prediction 

# Add the Brochure generator to Gradio

In [31]:
from scraper import fetch_website_contents

In [32]:
system_message = """You are an assistant that analyses the contents of a company website
landing page and creates a short brochure about the company for prospective customers, 
investors and recruits.
Respond in markdown without code blocks. """

In [34]:
def stream_brochure(company_name, url, model):
    yield ""
    prompt = f"Please generate a company brochure for {company_name}. Here is the landing page: \n"
    prompt += fetch_website_contents(url)
    if model == "GPT":
        result = stream_gpt(prompt)
    else:
        raise ValueError("Invalid Model")
    yield from result

In [36]:
COMPANY_NAME = "PPFAS"
URL = "https://amc.ppfas.com/index.php"

name_input = gr.Textbox(label="Company Name:")
url_input = gr.Textbox(label="URL")
model_selector = gr.Dropdown(["GPT", "Claude"], value="GPT", label="Select Model")
message_output = gr.Markdown("Response: ")

view = gr.Interface(
    title="Brochure Generator",
    fn=stream_brochure,
    inputs=[name_input, url_input, model_selector],
    outputs=[message_output],
    flagging_mode="never",
    examples=[
        [COMPANY_NAME, URL, "GPT"]
    ]
)

view.launch()

* Running on local URL:  http://127.0.0.1:7866
* To create a public link, set `share=True` in `launch()`.
